In [2]:
from datasets import load_dataset
import torch
import numpy as np
import chess
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

In [3]:
dataset = load_dataset(
    "mateuszgrzyb/lichess-stockfish-normalized",
    split="train",
    cache_dir="./training_data"
)

Loading dataset shards:   0%|          | 0/35 [00:00<?, ?it/s]

In [4]:
split1 = dataset.train_test_split(test_size=0.2, seed=42)

train_ds = split1["train"]
temp_ds  = split1["test"]

# second split: val vs test
split2 = temp_ds.train_test_split(test_size=0.5, seed=42)

val_ds   = split2["train"]
test_ds  = split2["test"]

In [5]:
#see https://official-stockfish.github.io/docs/nnue-pytorch-wiki/docs/nnue.html#halfkp
PIECE_MAP = {
    chess.PAWN: 0,
    chess.KNIGHT: 1,
    chess.BISHOP: 2,
    chess.ROOK: 3,
    chess.QUEEN: 4,
}

# using the same formula provided by the stockfish site
def halfkp_index(piece, piece_square, king_square):
    piece_type = PIECE_MAP[piece.piece_type]
    piece_color = 0 if piece.color == chess.WHITE else 1
    p_idx = piece_type * 2 + piece_color
    return piece_square + (p_idx + king_square * 10) * 64

def extract_halfkp(board, perspective):
    king_sq = board.king(perspective)
    feats = []

    for sq, piece in board.piece_map().items():
        if piece.piece_type == chess.KING:
            continue
        feats.append(halfkp_index(piece, sq, king_sq))

    return feats


In [6]:
NUM_FEATURES = 40960 # 64*64*5*2 (Each feature is a tuple (our_king_square, piece_square, piece_type, piece_color))

In [7]:
class ChessDataset(Dataset):
    def __init__(self, data, max_samples=None):
        self.ds = data

        if max_samples is not None:
            self.ds = self.ds.select(range(min(max_samples, len(self.ds))))

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]

        board = chess.Board(item["fen"])

        white_feats = extract_halfkp(board, chess.WHITE)
        black_feats = extract_halfkp(board, chess.BLACK)

        stm = 1.0 if board.turn == chess.WHITE else 0.0

        cp = item["cp"] or 0
        cp = max(min(cp, 1000), -1000) / 1000.0

        return {
            "white": torch.tensor(white_feats, dtype=torch.long),
            "black": torch.tensor(black_feats, dtype=torch.long),
            "stm": torch.tensor(stm, dtype=torch.float32),
            "target": torch.tensor([cp], dtype=torch.float32)
        }

In [8]:
def collate_fn(batch):

    white_offsets = []
    black_offsets = []

    white_flat = []
    black_flat = []

    stm = []
    target = []

    for i, b in enumerate(batch):

        w = b["white"]
        b_ = b["black"]

        # 🔥 FORCE correct dtype (critical for embeddings)
        w = torch.as_tensor(w, dtype=torch.long)
        b_ = torch.as_tensor(b_, dtype=torch.long)

        white_flat.append(w)
        black_flat.append(b_)

        white_offsets.append(torch.full((w.numel(),), i, dtype=torch.long))
        black_offsets.append(torch.full((b_.numel(),), i, dtype=torch.long))

        stm.append(torch.as_tensor(b["stm"], dtype=torch.float32))
        target.append(torch.as_tensor(b["target"], dtype=torch.float32))

    return {
        "white_idx": torch.cat(white_flat),
        "black_idx": torch.cat(black_flat),
        "white_batch": torch.cat(white_offsets),
        "black_batch": torch.cat(black_offsets),
        "stm": torch.stack(stm),
        "target": torch.stack(target),
    }

In [ ]:
class Accumulator:
    def __init__(self, white, black):
        self.white = white
        self.black = black

    def clone(self):
        return Accumulator(self.white.clone(), self.black.clone())

class NNUE(nn.Module):
    def __init__(self, num_features=40960):
        super().__init__()

        self.white_emb = nn.Embedding(num_features, 128)
        self.black_emb = nn.Embedding(num_features, 128)

        self.fc1 = nn.Linear(256, 128)
        self.fc2 = nn.Linear(128, 64)
        self.out = nn.Linear(64, 1)

        self.act = nn.ReLU()

    def segment_sum(self, emb, batch_idx, batch_size):
        out = torch.zeros(batch_size, emb.size(1), device=emb.device)
        return out.index_add_(0, batch_idx, emb)

    def forward(self, white_idx, black_idx, white_batch, black_batch, stm):

        B = stm.size(0)

        w_emb = self.white_emb(white_idx)
        b_emb = self.black_emb(black_idx)

        # 🔥 vectorized accumulation (NO LOOP)
        w = self.segment_sum(w_emb, white_batch, B)
        b = self.segment_sum(b_emb, black_batch, B)

        x = torch.cat([w, b], dim=1)

        flip = (stm == 0).view(-1, 1)
        x_flipped = torch.cat([b, w], dim=1)
        x = torch.where(flip, x_flipped, x)

        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        return self.out(x)
    
    def init_accumulator(self, board):
        white_feats = extract_halfkp(board, chess.WHITE)
        black_feats = extract_halfkp(board, chess.BLACK)

        device = next(self.parameters()).device

        w_idx = torch.tensor(white_feats, dtype=torch.long, device=device)
        b_idx = torch.tensor(black_feats, dtype=torch.long, device=device)

        acc = Accumulator(
            self.white_emb(w_idx).sum(dim=0),
            self.black_emb(b_idx).sum(dim=0),
        )

        return acc
    
    def evaluate_acc(self, acc, stm):
        w = acc.white
        b = acc.black

        x = torch.cat([w, b], dim=0)

        if stm == 0:
            x = torch.cat([b, w], dim=0)

        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        return self.out(x)
    

NameError: name 'nn' is not defined

In [10]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(device)

mps


In [11]:
BATCH_SIZE = 128
train_data = ChessDataset(train_ds, max_samples=100000)
val_data   = ChessDataset(val_ds, max_samples=20000)


train_loader = DataLoader(
    train_data,
    batch_size=128,
    shuffle=True,
    collate_fn=collate_fn,
    pin_memory=True
)

val_loader = DataLoader(
    val_data,
    batch_size=128,
    shuffle=False,
    collate_fn=collate_fn,
    pin_memory=True
)

In [12]:
model = NNUE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

In [13]:
EPOCHS = 5

for epoch in range(EPOCHS):

    # --------------------
    # TRAIN
    # --------------------
    model.train()
    train_loss = 0.0
    train_samples = 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [TRAIN]")

    for batch in train_bar:

        white_idx = batch["white_idx"].to(device)
        black_idx = batch["black_idx"].to(device)
        white_b   = batch["white_batch"].to(device)
        black_b   = batch["black_batch"].to(device)

        stm    = batch["stm"].to(device)
        target = batch["target"].to(device)

        # forward pass
        pred = model(white_idx, black_idx, white_b, black_b, stm)

        loss = loss_fn(pred, target)

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        batch_size = target.size(0)
        train_loss += loss.item() * batch_size
        train_samples += batch_size

        train_bar.set_postfix(loss=loss.item())

    train_loss /= train_samples

    # --------------------
    # VALIDATION
    # --------------------
    model.eval()
    val_loss = 0.0
    val_samples = 0

    val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1} [VAL]")

    with torch.no_grad():
        for batch in val_bar:

            white_idx = batch["white_idx"].to(device)
            black_idx = batch["black_idx"].to(device)
            white_b   = batch["white_batch"].to(device)
            black_b   = batch["black_batch"].to(device)

            stm    = batch["stm"].to(device)
            target = batch["target"].to(device)

            pred = model(white_idx, black_idx, white_b, black_b, stm)
            loss = loss_fn(pred, target)

            batch_size = target.size(0)
            val_loss += loss.item() * batch_size
            val_samples += batch_size

            val_bar.set_postfix(loss=loss.item())

    val_loss /= val_samples

    print(
        f"\nEpoch {epoch+1} Summary | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f}\n"
    )

Epoch 1 [VAL]: 100%|██████████| 157/157 [00:08<00:00, 17.55it/s, loss=0.0761]



Epoch 1 Summary | Train Loss: 0.098396 | Val Loss: 0.095189



Epoch 2 [VAL]: 100%|██████████| 157/157 [00:04<00:00, 37.49it/s, loss=0.0835]



Epoch 2 Summary | Train Loss: 0.090127 | Val Loss: 0.095616



Epoch 3 [VAL]: 100%|██████████| 157/157 [00:04<00:00, 38.13it/s, loss=0.0704]



Epoch 3 Summary | Train Loss: 0.081123 | Val Loss: 0.099387



Epoch 4 [VAL]: 100%|██████████| 157/157 [00:04<00:00, 37.97it/s, loss=0.0951]



Epoch 4 Summary | Train Loss: 0.066406 | Val Loss: 0.106893



Epoch 5 [VAL]: 100%|██████████| 157/157 [00:04<00:00, 37.89it/s, loss=0.0886]


Epoch 5 Summary | Train Loss: 0.051510 | Val Loss: 0.111463



In [18]:
test_data  = ChessDataset(test_ds, max_samples=100000)
test_loader = DataLoader(
    test_data,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn
)

model.eval()

test_loss = 0.0
test_mae = 0.0
test_samples = 0

test_bar = tqdm(test_loader, desc="TEST")

with torch.no_grad():
    for batch in test_bar:

        white_idx = batch["white_idx"].to(device)
        black_idx = batch["black_idx"].to(device)
        white_b   = batch["white_batch"].to(device)
        black_b   = batch["black_batch"].to(device)

        stm    = batch["stm"].to(device)
        target = batch["target"].to(device)

        pred = model(white_idx, black_idx, white_b, black_b, stm)

        loss = loss_fn(pred, target)

        batch_size = target.size(0)

        test_loss += loss.item() * batch_size
        test_mae  += torch.abs(pred - target).sum().item()
        test_samples += batch_size

        test_bar.set_postfix(loss=loss.item())

test_loss /= test_samples
test_mae  /= test_samples

print("\nFINAL TEST RESULTS")
print(f"Test MSE: {test_loss:.6f}")
print(f"Test MAE: {test_mae:.6f}")

TEST: 100%|██████████| 1563/1563 [00:24<00:00, 62.87it/s, loss=0.0897]


FINAL TEST RESULTS
Test MSE: 0.110109
Test MAE: 0.227610


In [19]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "epoch": epoch,
    "loss": train_loss
}, "model_new_checkpoint.pth")